In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Resolve the data path whether the notebook runs from notebooks/ or the repo root.
CSV = Path.cwd().parent / "data" / "processed" / "standardized" / "entity_crosswalk.csv"
if not CSV.exists():
    CSV = Path("data") / "processed" / "standardized" / "entity_crosswalk.csv"

# Keep the existing CSV variable if it already points to a valid relative file.
if not CSV.exists():
    # Try standard locations from the current working directory and its parents.
    candidates = [
        Path.cwd() / "data" / "processed" / "standardized" / "entity_crosswalk.csv",
        Path.cwd().parent / "data" / "processed" / "standardized" / "entity_crosswalk.csv",
    ]
    for parent in Path.cwd().parents:
        candidates.append(parent / "data" / "processed" / "standardized" / "entity_crosswalk.csv")

    CSV = next((p for p in candidates if p.exists()), CSV)

# If the path is still not found, search recursively for the file in the workspace.
if not CSV.exists():
    matches = list(Path.cwd().rglob("entity_crosswalk.csv"))
    if matches:
        CSV = matches[0]
    else:
        raise FileNotFoundError(
            "Could not find 'entity_crosswalk.csv'. "
            "Generate the standardized file in "
            "'data/processed/standardized/entity_crosswalk.csv' first."
        )

df = pd.read_csv(CSV)
print("Loaded:", CSV)
print("Shape:", df.shape)
df.head()

Loaded: c:\Users\bedoor.alsulami\AppData\Local\Packages\PythonSoftwareFoundation.PythonManager_3847v3x7pw1km\AppData\saudi-Digital-Concierge\data\processed\standardized\entity_crosswalk.csv
Shape: (10382, 7)


,canonical_id,source,source_entity_id,entity_type,name,city,region
0,ASR_000001,booking_kaggle,hotels_00732,hotel,AL Faridah Furnished Units,Khamis Mushait,Asir
1,ASR_000002,booking_kaggle,hotels_00428,hotel,Abha Airport Hotel,Abha,Asir
2,ASR_000003,booking_kaggle,hotels_00040,hotel,Abha Palace Hotel,Abha,Asir
3,ASR_000004,booking_kaggle,hotels_00293,hotel,Abha Sky (Families Only),Abha,Asir
4,ASR_000005,booking_kaggle,hotels_00804,hotel,Al Afaq Alraqi Furnished Apartments (Families ...,Abha,Asir


In [15]:
location_cols = [c for c in ["city", "region", "location"] if c in df.columns]

if location_cols:
    for col in location_cols:
        vc = df[col].value_counts(dropna=False)
        print(f"=== {col} — {df[col].nunique(dropna=True)} unique ===")
        print(vc.head(30).to_string(), "\n")
else:
    print("No city/location column detected — check Q2 fields.")

=== city — 117 unique ===
city
Riyadh            9299
Jeddah             245
Al Khobar           97
Makkah              83
Dammam              74
Abha                42
Madinah             40
Buraydah            39
Taif                37
Yanbu               32
Jazan               27
Tabuk               27
Hail                25
Khamis Mushait      22
NaN                 21
Jubail              19
Al Ahsa             14
Unayzah             10
Najran               8
Sharurah             8
Al Kharj             7
Dhahran              7
Al Bahah             6
Al Muzahimiyah       6
Half Moon Bay        5
Al Khafji            5
Diriyah              5
Ţurayf               4
Hafr Al Baten        4
Al Hada              4 

=== region — 12 unique ===
region
Riyadh              9329
Makkah               381
Eastern Province     230
NaN                  129
Madinah               76
Asir                  72
Qassim                52
Tabuk                 29
Jazan                 27
Hail              

In [2]:
fields = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(dtype) for dtype in df.dtypes],
    "nunique": [int(df[col].nunique(dropna=True)) for col in df.columns],
    "example": [
        df[col].dropna().iloc[0] if df[col].notna().any() else None
        for col in df.columns
    ],
})
fields

,column,dtype,nunique,example
0,canonical_id,str,10027,ASR_000001
1,source,str,3,booking_kaggle
2,source_entity_id,str,10382,hotels_00732
3,entity_type,str,3,hotel
4,name,str,6519,AL Faridah Furnished Units
5,city,str,117,Khamis Mushait
6,region,str,12,Asir
